# DeePoo EfficientDet-Lite 320x320 — TF SavedModel to TFLite FP16 and INT8 Conversion

This notebook converts the trained EfficientDet-Lite0 TF SavedModel model to TFLite format in FP16 precision and with INT8 quantization for Android deployment.

**Input**: Zipped model in TensorFlow SavedModel format   
**Output**: TFLite FP16 model and INT8 quantized model  
**Dataset**: Training images for calibration  
**Environment**: Google Colab with GPU

## Conversion Process

1. Convert TensorFlow → TFLite with FP16 precision
2. Test and benchmark the TFLite model with FP16 precision
3. Convert TensorFlow → TFLite with INT8 quantization
4. Test and benchmark the TFLite model with INT8 quatization
5. Compare the performance of the FP16 and INT8 models

## 1. Install Dependencies

In [ ]:
!pip install pycocotools opencv-python matplotlib tqdm

## 2. Import Libraries

In [ ]:
import os
from pathlib import Path

import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import json

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from tqdm import tqdm


## 3. Setup Paths

In [ ]:
# SavedModel Configuration
SAVEDMODEL_ZIP_PATH = '/content/DeePoo-EfficientDet-Lite0-251114.zip'
SAVEDMODEL_EXTRACT_DIR = Path('/content/saved_model')

# Dataset Configuration
DATASET_EXTRACT_PATH = Path('/content/dataset')
DATASET_ZIP_PATH = '/content/poo_base_640x640.zip'
DATASET_NAME = os.path.splitext(os.path.basename(DATASET_ZIP_PATH))[0]
DATASET_ROOT = DATASET_EXTRACT_PATH / DATASET_NAME

In [ ]:
# Create necessary directories
os.makedirs(SAVEDMODEL_EXTRACT_DIR, exist_ok=True)
os.makedirs(DATASET_EXTRACT_PATH, exist_ok=True)

## 3.A Extract SavedModel

In [ ]:
import zipfile

print("📦 Extracting SavedModel...")
print(f"   Source: {SAVEDMODEL_ZIP_PATH}")
print(f"   Destination: {SAVEDMODEL_EXTRACT_DIR}")

with zipfile.ZipFile(SAVEDMODEL_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(SAVEDMODEL_EXTRACT_DIR)

print("✅ SavedModel extracted successfully!")

# Inspect extracted structure
for root, dirs, files in os.walk(SAVEDMODEL_EXTRACT_DIR):
    level = root.replace(str(SAVEDMODEL_EXTRACT_DIR), "").count(os.sep)
    indent = "   " * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    subindent = "   " * (level + 1)
    for f in files:
        print(f"{subindent}📄 {f}")
    if level >= 2:  # avoid printing huge trees
        break

# Try common pattern: zip contains a folder named 'saved_model'
TF_SAVEDMODEL_DIR = SAVEDMODEL_EXTRACT_DIR / 'saved_model'

print(f"\n🔎 TF SavedModel directory candidate: {TF_SAVEDMODEL_DIR}")
print(f"   Exists: {TF_SAVEDMODEL_DIR.exists()}")

## 4. Device Setup and Configuration

In [ ]:
# Check GPU availability and configure
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        # Enable memory growth to prevent TensorFlow from allocating all GPU memory
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        # Set the GPU as the default device
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"✅ GPU Configuration:")
        print(f"   Physical GPUs: {len(gpus)}")
        print(f"   Logical GPUs: {len(logical_gpus)}")
        print(f"   GPU Name: {gpus[0].name}")

        # Get GPU details
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        if 'device_name' in gpu_details:
            print(f"   GPU Model: {gpu_details['device_name']}")

        DEVICE = '/GPU:0'

    except RuntimeError as e:
        print(f"⚠️  GPU setup error: {e}")
        DEVICE = '/CPU:0'
else:
    print("⚠️  No GPU detected, using CPU")
    DEVICE = '/CPU:0'

print(f"\n🖥️  Training Device: {DEVICE}")

# Configure mixed precision for faster training (if GPU available)
if gpus:
    try:
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)
        print(f"✅ Mixed precision enabled: {policy.name}")
        print(f"   Compute dtype: {policy.compute_dtype}")
        print(f"   Variable dtype: {policy.variable_dtype}")
    except:
        print("⚠️  Mixed precision not available, using float32")

# Display TensorFlow configuration
print(f"\n📊 TensorFlow Configuration:")
print(f"   Version: {tf.__version__}")
print(f"   Eager Execution: {tf.executing_eagerly()}")
print(f"   Built with CUDA: {tf.test.is_built_with_cuda()}")

## 5. Extract Dataset

In [ ]:
from re import I
import zipfile

print("📂 Extracting dataset...")
print(f"   Source: {DATASET_ZIP_PATH}")
print(f"   Destination: {DATASET_EXTRACT_PATH}")

# Extract the dataset
with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DATASET_EXTRACT_PATH)

print("✅ Dataset extracted successfully!")

print(f"\n📁 Dataset structure:")
for item in sorted(DATASET_ROOT.iterdir()):
    if item.is_dir():
        num_files = len(list(item.rglob('*')))
        print(f"   📂 {item.name}/ ({num_files} items)")
    else:
        print(f"   📄 {item.name}")

# COCO annotation files
ANNOTATIONS_DIR = DATASET_ROOT / 'annotations'
TRAIN_ANNOTATIONS = ANNOTATIONS_DIR / 'instances_train.json'
VAL_ANNOTATIONS = ANNOTATIONS_DIR / 'instances_val.json'
TEST_ANNOTATIONS = ANNOTATIONS_DIR / 'instances_test.json'

# Image files
IMAGES_DIR = DATASET_ROOT / 'images'
TRAIN_IMAGES_DIR = IMAGES_DIR / 'train'
VAL_IMAGES_DIR = IMAGES_DIR / 'val'
TEST_IMAGES_DIR = IMAGES_DIR / 'test'

print(f"\n📸 Image directories:")
print(f"   Train: {TRAIN_IMAGES_DIR}")
print(f"   Val: {VAL_IMAGES_DIR}")
print(f"   Test: {TEST_IMAGES_DIR}")

## 6. Explore Dataset

### Load dataset

In [ ]:
print("📊 Loading COCO annotations...")

# Load training annotations
coco_train = COCO(TRAIN_ANNOTATIONS)
print(f"✅ Training annotations loaded")

# Load validation annotations
coco_val = COCO(VAL_ANNOTATIONS)
print(f"✅ Validation annotations loaded")

# Load test annotations
coco_test = COCO(TEST_ANNOTATIONS)
print(f"✅ Test annotations loaded")

# Get dataset statistics
train_img_ids = coco_train.getImgIds()
val_img_ids = coco_val.getImgIds()
test_img_ids = coco_test.getImgIds()

train_cat_ids = coco_train.getCatIds()
val_cat_ids = coco_val.getCatIds()
test_cat_ids = coco_test.getCatIds()

train_categories = coco_train.loadCats(train_cat_ids)
val_categories = coco_val.loadCats(val_cat_ids)
test_categories = coco_test.loadCats(test_cat_ids)

# Count annotations
train_ann_ids = coco_train.getAnnIds()
val_ann_ids = coco_val.getAnnIds()
test_ann_ids = coco_test.getAnnIds()

print(f"\n📈 Dataset Statistics:")
print(f"   {'='*50}")
print(f"   Training Set:")
print(f"      Images: {len(train_img_ids)}")
print(f"      Annotations: {len(train_ann_ids)}")
print(f"      Categories: {len(train_categories)}")
print(f"   {'='*50}")
print(f"   Validation Set:")
print(f"      Images: {len(val_img_ids)}")
print(f"      Annotations: {len(val_ann_ids)}")
print(f"      Categories: {len(val_categories)}")
print(f"   {'='*50}")
print(f"   Test Set:")
print(f"      Images: {len(test_img_ids)}")
print(f"      Annotations: {len(test_ann_ids)}")
print(f"      Categories: {len(test_categories)}")

# Display category information
print(f"\n🏷️  Categories:")
for cat in train_categories:
    # Count annotations for this category
    ann_ids = coco_train.getAnnIds(catIds=[cat['id']])
    print(f"   ID {cat['id']}: {cat['name']} ({len(ann_ids)} annotations)")

# Calculate average annotations per image
avg_train_ann = len(train_ann_ids) / len(train_img_ids) if train_img_ids else 0
avg_val_ann = len(val_ann_ids) / len(val_img_ids) if val_img_ids else 0
avg_test_ann = len(test_ann_ids) / len(test_img_ids) if val_img_ids else 0

print(f"\n📊 Annotation Density:")
print(f"   Training: {avg_train_ann:.2f} annotations per image")
print(f"   Validation: {avg_val_ann:.2f} annotations per image")
print(f"   Test: {avg_test_ann:.2f} annotaions per image")

# Sample image information
if train_img_ids:
    sample_img = coco_train.loadImgs(train_img_ids[0])[0]
    print(f"\n🖼️  Sample Image Info:")
    print(f"   File: {sample_img['file_name']}")
    print(f"   Size: {sample_img['width']}x{sample_img['height']}")
    print(f"   ID: {sample_img['id']}")

# Store for later use
COCO_TRAIN = coco_train
COCO_VAL = coco_val
COCO_TEST = coco_test
TRAIN_IMG_IDS = train_img_ids
VAL_IMG_IDS = val_img_ids
TEST_IMG_IDS = test_img_ids
CATEGORY_NAMES = [cat['name'] for cat in train_categories]
CATEGORY_IDS = [cat['id'] for cat in train_categories]

print(f"\n✅ Dataset exploration complete!")

### Visualize samples with annotations

In [ ]:
def visualize_coco_samples(coco, img_ids, images_dir, num_samples=4, cols=2):
    """Visualize sample images with bounding box annotations"""

    num_samples = min(num_samples, len(img_ids))
    rows = (num_samples + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 7*rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = axes.reshape(rows, cols)

    # Randomly sample images
    sample_ids = random.sample(img_ids, num_samples)

    for idx, img_id in enumerate(sample_ids):
        row = idx // cols
        col = idx % cols
        ax = axes[row, col]

        # Load image info
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(images_dir, img_info['file_name'])

        # Load and display image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)

        # Get annotations for this image
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        # Draw bounding boxes
        for ann in anns:
            bbox = ann['bbox']  # [x, y, width, height]
            x, y, w, h = bbox

            # Get category info
            cat = coco.loadCats(ann['category_id'])[0]

            # Draw rectangle
            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2,
                edgecolor='fuchsia',
                facecolor='none'
            )
            ax.add_patch(rect)

            # Add label
            label = f"{cat['name']}"
            ax.text(
                x, y - 5,
                label,
                color='white',
                fontsize=10,
                bbox=dict(facecolor='fuchsia', alpha=0.7, edgecolor='none', pad=2)
            )

        ax.set_title(f"{img_info['file_name']}\n{len(anns)} annotations", fontsize=10)
        ax.axis('off')

    # Hide empty subplots
    for idx in range(num_samples, rows * cols):
        row = idx // cols
        col = idx % cols
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()

# Visualize training samples
print("🖼️  Training Set Samples:")
visualize_coco_samples(COCO_TRAIN, TRAIN_IMG_IDS, TRAIN_IMAGES_DIR, num_samples=3, cols=3)

# Visualize validation samples
print("\n🖼️  Validation Set Samples:")
visualize_coco_samples(COCO_VAL, VAL_IMG_IDS, VAL_IMAGES_DIR, num_samples=3, cols=3)

# Visualize test samples
print("\n🖼️  Test Set Samples:")
visualize_coco_samples(COCO_TEST, TEST_IMG_IDS, TEST_IMAGES_DIR, num_samples=3, cols=3)

## 6. Convert TensorFlow SavedModel to TFLite FP16

In [ ]:
%%time
"""
Convert TensorFlow SavedModel to TFLite FP16, allowing TF Select ops.
"""

# Use the SavedModel directory determined above
tf_model_dir = TF_SAVEDMODEL_DIR
assert tf_model_dir.exists(), f"SavedModel directory does not exist: {tf_model_dir}"

print("Converting TensorFlow SavedModel to TFLite FP16...")
print("=" * 60)
print(f"Input SavedModel dir: {tf_model_dir}")

# Create TFLite converter
converter = tf.lite.TFLiteConverter.from_saved_model(str(tf_model_dir))

# Enable default optimizations
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Allow both native TFLite ops and TF Select ops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,      # enable built-in TFLite ops
    tf.lite.OpsSet.SELECT_TF_OPS,        # enable TF Select (fallback to TF kernels)
]

# Request FP16 storage for weights
converter.target_spec.supported_types = [tf.float16]

print("🔄 Running FP16 conversion with TF Select ops enabled...")
tflite_fp16_model = converter.convert()

# Save FP16 model
tflite_fp16_path = SAVEDMODEL_EXTRACT_DIR / 'deepoo_efficientdet_lite0_fp16.tflite'
with open(tflite_fp16_path, 'wb') as f:
    f.write(tflite_fp16_model)

model_size_mb = len(tflite_fp16_model) / (1024 * 1024)
print(f"\n✅ TFLite FP16 model saved to: {tflite_fp16_path}")
print(f"   Model size: {model_size_mb:.2f} MB")

## 7. Inspect FP16 TFLite Model

In [ ]:
# Load TFLite FP16 model
interpreter_fp16 = tf.lite.Interpreter(model_path=str(tflite_fp16_path))
interpreter_fp16.allocate_tensors()

input_details_fp16 = interpreter_fp16.get_input_details()
output_details_fp16 = interpreter_fp16.get_output_details()

print("TFLite FP16 Model Information")
print("=" * 60)

print("\nInput Details:")
for i, inp in enumerate(input_details_fp16):
    print(f"  Input {i}:")
    print(f"    Name: {inp['name']}")
    print(f"    Shape: {inp['shape']}")
    print(f"    Type: {inp['dtype']}")

print("\nOutput Details:")
for i, out in enumerate(output_details_fp16):
    print(f"  Output {i}:")
    print(f"    Name: {out['name']}")
    print(f"    Shape: {out['shape']}")
    print(f"    Type: {out['dtype']}")

print("\n✅ TFLite FP16 model loaded successfully")

## 8. Runt TFLite FP16 Inference on Single Test Image

In [ ]:
IMAGE_SIZE = 320  # EfficientDet-Lite0 input size

# Pick one test image
test_img_path = random.choice(list(TEST_IMAGES_DIR.glob("*.jpg")))
print(f"Using test image: {test_img_path}")

# Load & preprocess
img_bgr = cv2.imread(str(test_img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (IMAGE_SIZE, IMAGE_SIZE))

# Input type and shape
input_details = interpreter_fp16.get_input_details()
output_details = interpreter_fp16.get_output_details()
print("Original input shape from interpreter:", input_details[0]["shape"])

# Resize input tensor to [1, 320, 320, 3]
input_index = input_details[0]["index"]
interpreter_fp16.resize_tensor_input(input_index, [1, IMAGE_SIZE, IMAGE_SIZE, 3])
interpreter_fp16.allocate_tensors()  # re-allocate after resize

# Prepare input according to the model's expected dtype (uint8)
input_details = interpreter_fp16.get_input_details()
output_details = interpreter_fp16.get_output_details()
print("New input shape:", input_details[0]["shape"])
print("Input dtype:", input_details[0]["dtype"])

input_data = img_resized.astype(input_details[0]["dtype"])
input_data = np.expand_dims(input_data, axis=0)  # [1, H, W, C]

# Run inference
interpreter_fp16.set_tensor(input_details[0]["index"], input_data)
interpreter_fp16.invoke()

detections = interpreter_fp16.get_tensor(output_details[0]["index"])
print("Detections shape:", detections.shape)
print("First detection row:", detections[0, 0])

## 9. Evaluate TFLite FP16 Model Quality

## 10. Prepare Calibration Data for INT8 Quantization

In [ ]:
# Use a subset of training images for calibration
MAX_CALIBRATION_IMAGES = 200  # you can increase if Colab time allows
calibration_img_paths = list(TRAIN_IMAGES_DIR.glob("*.jpg"))[:MAX_CALIBRATION_IMAGES]

print(f"Using {len(calibration_img_paths)} images for INT8 calibration")

IMAGE_SIZE = 320  # ensure consistent

def load_image_for_model(path, image_size=IMAGE_SIZE):
    """Load image, resize to model input size, keep uint8."""
    img_bgr = cv2.imread(str(path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (image_size, image_size))
    return img_resized.astype(np.uint8)

def representative_dataset_gen():
    """Generator function for TFLite INT8 calibration."""
    for img_path in tqdm(calibration_img_paths, desc="Calibration images"):
        img = load_image_for_model(img_path)
        img = np.expand_dims(img, axis=0)  # [1, H, W, C]
        yield [img]

print("✅ Representative dataset generator defined")

## 11. Convert TensorFlow SavedModel to TFLite INT8

In [ ]:
%%time

tf_model_dir = TF_SAVEDMODEL_DIR
assert tf_model_dir.exists(), f"SavedModel directory does not exist: {tf_model_dir}"

print("Converting TensorFlow SavedModel to TFLite INT8...")
print("=" * 60)
print(f"Input SavedModel dir: {tf_model_dir}")
print(f"Calibration images: {len(calibration_img_paths)}")
print("\n⏳ This may take several minutes (calibration)...\n")

converter = tf.lite.TFLiteConverter.from_saved_model(str(tf_model_dir))

# Enable full integer quantization with calibration
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Allow INT8 built-ins + TF Select ops (fallback)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

# Try full INT8 I/O; if this fails later we can relax to float I/O
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

print("🔄 Running INT8 quantization with calibration...")
tflite_int8_model = converter.convert()

# Save INT8 model
tflite_int8_path = SAVEDMODEL_EXTRACT_DIR / "deepoo_efficientdet_lite0_int8.tflite"
with open(tflite_int8_path, "wb") as f:
    f.write(tflite_int8_model)

model_size_mb = len(tflite_int8_model) / (1024 * 1024)
print(f"\n✅ TFLite INT8 model saved to: {tflite_int8_path}")
print(f"   Model size: {model_size_mb:.2f} MB")

## 11. Test TFLite INT8 Model

In [ ]:
interpreter_int8 = tf.lite.Interpreter(model_path=str(tflite_int8_path))
interpreter_int8.allocate_tensors()

input_details_int8 = interpreter_int8.get_input_details()
output_details_int8 = interpreter_int8.get_output_details()

print("TFLite INT8 Model Information:")
print("=" * 60)

print("\nInput Details:")
for i, inp in enumerate(input_details_int8):
    print(f"  Input {i}:")
    print(f"    Name: {inp['name']}")
    print(f"    Shape: {inp['shape']}")
    print(f"    Type: {inp['dtype']}")
    print(f"    Quantization: scale={inp['quantization'][0]}, zero_point={inp['quantization'][1]}")

print("\nOutput Details:")
for i, out in enumerate(output_details_int8):
    print(f"  Output {i}:")
    print(f"    Name: {out['name']}")
    print(f"    Shape: {out['shape']}")
    print(f"    Type: {out['dtype']}")
    print(f"    Quantization: scale={out['quantization'][0]}, zero_point={out['quantization'][1]}")

print("\n✅ TFLite INT8 model loaded successfully")

## 12. Run TFLite INT8 Inference on Single Test Image

In [ ]:
# INT8 inference on a single test image

# Ensure IMAGE_SIZE and TEST_IMAGES_DIR are defined (from previous cells)
IMAGE_SIZE = 320  # EfficientDet-Lite0
assert len(list(TEST_IMAGES_DIR.glob("*.jpg"))) > 0, "No test images found"

test_img_path_int8 = random.choice(list(TEST_IMAGES_DIR.glob("*.jpg")))
print(f"Using test image for INT8: {test_img_path_int8}")

# Load & preprocess image as uint8
img_bgr = cv2.imread(str(test_img_path_int8))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (IMAGE_SIZE, IMAGE_SIZE))

# Get interpreter details
input_details_int8 = interpreter_int8.get_input_details()
output_details_int8 = interpreter_int8.get_output_details()

print("Original INT8 input shape:", input_details_int8[0]["shape"])
print("INT8 input dtype:", input_details_int8[0]["dtype"])

# Resize input tensor to [1, H, W, C]
input_index_int8 = input_details_int8[0]["index"]
interpreter_int8.resize_tensor_input(input_index_int8, [1, IMAGE_SIZE, IMAGE_SIZE, 3])
interpreter_int8.allocate_tensors()  # re-allocate after resize

# Refresh details after resize
input_details_int8 = interpreter_int8.get_input_details()
output_details_int8 = interpreter_int8.get_output_details()

print("Resized INT8 input shape:", input_details_int8[0]["shape"])
print("INT8 input dtype (after resize):", input_details_int8[0]["dtype"])

# Prepare input tensor
input_data_int8 = img_resized.astype(input_details_int8[0]["dtype"])
input_data_int8 = np.expand_dims(input_data_int8, axis=0)  # [1, H, W, C]

# Run inference
interpreter_int8.set_tensor(input_details_int8[0]["index"], input_data_int8)
interpreter_int8.invoke()

detections_int8_q = interpreter_int8.get_tensor(output_details_int8[0]["index"])
print("Raw INT8 detections shape:", detections_int8_q.shape)
print("First detection (quantized) row:", detections_int8_q[0, 0])

# Dequantize outputs to float for readability
scale, zero_point = output_details_int8[0]["quantization"]
detections_int8 = (detections_int8_q.astype(np.float32) - zero_point) * scale

print("\nFirst detection (dequantized):", detections_int8[0, 0])

## 13. Evaluate INT8 Model Quality

In [ ]:
# Compute metrics for INT8 model
print("Computing INT8 model evaluation metrics...")
print("="*60)

tflite_int8_metrics = compute_metrics(tflite_int8_results, test_coco, iou_threshold=0.5, conf_threshold=0.3)

print("\n📊 TFLite INT8 Model Metrics (IoU=0.5, Conf=0.3):")
print(f"  Precision: {tflite_int8_metrics['precision']:.4f}")
print(f"  Recall:    {tflite_int8_metrics['recall']:.4f}")
print(f"  F1-Score:  {tflite_int8_metrics['f1_score']:.4f}")
print(f"  TP: {tflite_int8_metrics['tp']}, FP: {tflite_int8_metrics['fp']}, FN: {tflite_int8_metrics['fn']}")

print("\n📈 Quality Degradation vs Float Models:")
print(f"\n  vs ONNX (FP32):")
print(f"    ΔPrecision: {abs(tflite_int8_metrics['precision'] - onnx_metrics['precision']):.4f}")
print(f"    ΔRecall:    {abs(tflite_int8_metrics['recall'] - onnx_metrics['recall']):.4f}")
print(f"    ΔF1-Score:  {abs(tflite_int8_metrics['f1_score'] - onnx_metrics['f1_score']):.4f}")

print(f"\n  vs TF SavedModel (FP32):")
print(f"    ΔPrecision: {abs(tflite_int8_metrics['precision'] - tf_metrics['precision']):.4f}")
print(f"    ΔRecall:    {abs(tflite_int8_metrics['recall'] - tf_metrics['recall']):.4f}")
print(f"    ΔF1-Score:  {abs(tflite_int8_metrics['f1_score'] - tf_metrics['f1_score']):.4f}")

# Quality assessment
f1_degradation = abs(tflite_int8_metrics['f1_score'] - onnx_metrics['f1_score'])
if f1_degradation < 0.02:
    quality = "Excellent (minimal degradation)"
    icon = "🟢"
elif f1_degradation < 0.05:
    quality = "Good (acceptable degradation)"
    icon = "🟡"
elif f1_degradation < 0.10:
    quality = "Fair (noticeable degradation)"
    icon = "🟠"
else:
    quality = "Poor (significant degradation)"
    icon = "🔴"

print(f"\n{icon} INT8 Quantization Quality: {quality}")
print(f"   F1-Score degradation: {f1_degradation:.4f}")

print("="*60)

## 14. Compare TFLite FP16 and INT8 Models